<br><br><br>
## LightGBM 

In [88]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import json

# Load the dataset
data_path = "dataset.csv"
data = pd.read_csv(data_path)

data = data.loc[:, ~data.columns.str.contains("timestamp", case=False)]

# Shuffle the dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features (X) and labels (y)
X = data.drop(columns=["label"])  # Exclude 'label' and 'time_stamp_0'
y = data["label"]

# Encode the labels and get the mapping
y, label_mapping = pd.factorize(data["label"])  # 'y' gets numeric labels, 'label_mapping' holds original labels

# Save class index -> label mapping to JSON
label_dict = {i: label for i, label in enumerate(label_mapping)}
with open("class_labels.json", "w") as f:
    json.dump(label_dict, f, indent=4)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)


In [89]:

# Define parameters for LightGBM
params = {
    "objective": "multiclass",
    "num_class": len(pd.unique(y)),  # Number of classes
    "boosting_type": "gbdt",
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 5,
    "verbosity": -1,
    "random_state": 42,
     "min_data_in_leaf": 20
}

# Train the model
model = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=1200)

# Make predictions
y_pred = model.predict(X_test)
y_pred = [list(pred).index(max(pred)) for pred in y_pred]  # Convert probabilities to class indices

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model for deployment
#model.save_model("model_lightgbm.txt")
#print("Model saved successfully!")

Test Accuracy: 0.9808


In [67]:
def load_sample_as_array(json_path, expected_dim=None):
    import json
    import numpy as np

    with open(json_path, 'r') as f:
        data = json.load(f)

    arr = np.array(data, dtype=np.float64).reshape(1, -1)

    if expected_dim and arr.shape[1] != expected_dim:
        raise ValueError(f"Expected {expected_dim} features, got {arr.shape[1]}")

    np.set_printoptions(precision=6, suppress=True, linewidth=200)
    print("Sample input:")
    print(arr)
    
    return arr


In [8]:
import lightgbm as lgb



MODEL_PATH      = "model_lightgbm.txt"  # your saved model
# If you have a list of label names in order, you can put them here:
LABEL_NAMES     = [
    "alsalam_alikum",
    "good_morning",
    "wht_ur_name",
    "how_old_are_u",
    "how_are_you"
] 

model = lgb.Booster(model_file=MODEL_PATH)


In [71]:
arr = load_sample_as_array('test.json', 300)

Sample input:
[[   0.000012    1.          1.          1.          1.          1.         -0.706289   -0.56982     9.346963  -12.755725   38.610687   11.412214    0.110133    1.          1.          1.
     1.          1.          0.292093    2.801216   11.508927    2.099237  -87.526718   11.465649    0.218292    1.          1.          1.          1.          1.          2.542642    1.149217
    10.42196    -7.557252  -34.480916   45.267176    0.328265    1.          1.          1.          1.          1.          1.939303    0.876278   10.776302   -4.374046  -62.885496   82.251908
     0.438264    1.          1.          1.          1.          1.          3.828328    0.574608    8.281544   93.854962 -162.70229   152.442748    0.548241    1.          1.          1.
     1.          1.          4.860229   -0.708684    6.584055   49.40458   -95.59542   138.465649    0.658197    1.          1.          1.          1.          1.          7.807492   -0.383072
     5.566519  113.473282  -

In [82]:
X_test[0:1]

,time_stamp_0,flex_finger_0_0,flex_finger_0_1,flex_finger_0_2,flex_finger_0_3,flex_finger_0_4,acc_0_x,acc_0_y,acc_0_z,gyro_0_x,...,flex_finger_24_1,flex_finger_24_2,flex_finger_24_3,flex_finger_24_4,acc_24_x,acc_24_y,acc_24_z,gyro_24_x,gyro_24_y,gyro_24_z
275,3.143922,0,0,0,0,0,-9.303868,-0.701501,-0.316035,-7.503817,...,0,0,0,0,-9.212888,-0.837971,-0.909797,-13.572519,-2.885496,-2.687023


In [50]:
with open('test.json', 'r') as file:
    data = json.load(file)

In [58]:
import numpy as np
arr = np.array(data, dtype=np.float64).reshape(1, -1)

In [85]:
model.predict(X_test[0:1])

array([[0.      , 0.      , 0.      , 0.      , 0.999999]])

In [73]:
model.predict(arr)

array([[0.999965, 0.000004, 0.000018, 0.000002, 0.000011]])

In [25]:
X_test[0:1].

(1, 300)

In [63]:
X_test[0:1]

,time_stamp_0,flex_finger_0_0,flex_finger_0_1,flex_finger_0_2,flex_finger_0_3,flex_finger_0_4,acc_0_x,acc_0_y,acc_0_z,gyro_0_x,...,flex_finger_24_1,flex_finger_24_2,flex_finger_24_3,flex_finger_24_4,acc_24_x,acc_24_y,acc_24_z,gyro_24_x,gyro_24_y,gyro_24_z
275,3.143922,0,0,0,0,0,-9.303868,-0.701501,-0.316035,-7.503817,...,0,0,0,0,-9.212888,-0.837971,-0.909797,-13.572519,-2.885496,-2.687023


In [32]:
def calculate_accuracy(predicted, actual):
    if len(predicted) != len(actual):
        raise ValueError("Lists must be of the same length")

    correct = sum(p == a for p, a in zip(predicted, actual))
    accuracy = (correct / len(actual)) * 100

    return accuracy


In [33]:
calculate_accuracy(arr, X_test[0:1])

/tmp/ipykernel_10645/2006385826.py:15: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  correct = sum(p == a for p, a in zip(predicted, actual))


0.0

In [44]:
real_list[0:5]

[1.2159347534179688e-05, 1, 1, 1, 1]

In [45]:
aarr

NameError: name 'aarr' is not defined

In [43]:
X_test[0:1]

,time_stamp_0,flex_finger_0_0,flex_finger_0_1,flex_finger_0_2,flex_finger_0_3,flex_finger_0_4,acc_0_x,acc_0_y,acc_0_z,gyro_0_x,...,flex_finger_24_1,flex_finger_24_2,flex_finger_24_3,flex_finger_24_4,acc_24_x,acc_24_y,acc_24_z,gyro_24_x,gyro_24_y,gyro_24_z
275,3.143922,0,0,0,0,0,-9.303868,-0.701501,-0.316035,-7.503817,...,0,0,0,0,-9.212888,-0.837971,-0.909797,-13.572519,-2.885496,-2.687023
